# loadtest — interactive

Cell-by-cell version of the scripts in `scenarios/*.ts`, sharing the same logic from `lib/scenarios.ts`.

**Before running scenarios 1 or 3**: check whether your Firebase project is on Spark (free, hard quota stop, zero cost) or Blaze (pay-as-you-go, can bill you) — Console → Usage & billing.

**Reading results**: Server Action form posts always return HTTP 200, even on validation failure — status is not a success signal. The only ground truth is checking the Firestore `contactSubmissions` collection after each run.

In [ ]:
import { TARGET_URL, CONTACT_PATH, assertSafeTarget } from "./config.ts";
import * as scenarios from "./lib/scenarios.ts";

assertSafeTarget(TARGET_URL);
const pageUrl = new URL(CONTACT_PATH, TARGET_URL).toString();
console.log("target:", TARGET_URL);
console.log("contact page:", pageUrl);

## Scenario 1 — flood the contact form

No CAPTCHA/honeypot/rate limit today, so this should succeed 100% of the time. Each success is a real Firestore write + a real Resend email send — start small.

In [ ]:
const flood = await scenarios.floodContact(pageUrl, /* count */ 2, /* concurrency */ 2);
flood

## Scenario 2 — oversized payload

`name`/`details` have no `.max()` in `src/models/contactSubmission.ts`. Sends one large `details` field.

In [ ]:
const oversized = await scenarios.oversizedPayload(pageUrl, /* sizeKb */ 1024);
console.log("status:", oversized.status, "elapsed:", oversized.elapsedMs.toFixed(0), "ms");
oversized.bodySnippet

## Scenario 3 — concurrent race

Fires a burst all at once to probe the addDoc -> Resend.send -> updateDoc/deleteDoc rollback logic in `src/data/contactSubmissions.ts`. Check Firestore afterwards for any doc stuck in `emailStatus: 'pending'`.

In [ ]:
const raceStatuses = await scenarios.concurrentRace(pageUrl, /* count */ 10);
raceStatuses

## Scenario 4 — bypass client-side validation

Submits directly with no consent / bad email / too-short details. Should always be rejected server-side — this one should already pass.

In [ ]:
const bypassResults = await scenarios.bypassConsentCases(pageUrl);
for (const [label, result] of Object.entries(bypassResults)) {
  console.log(label, "->", result.status, result.bodySnippet.slice(0, 80));
}

## Scenario 5 — static page flood

No Firestore/Resend involved — only exercises the Cloudflare Workers free-tier request ceiling. Safe to scale up.

In [ ]:
const staticSummary = await scenarios.staticPageFlood(
  TARGET_URL,
  ["/", "/services", "/contact", "/about"],
  /* count */ 50,
  /* concurrency */ 10,
);
staticSummary